# RAG 项目核心代码导览

这个 notebook 用**已入库的真实数据**把项目的核心代码走一遍：数据契约 → 切分 → 检索 → 评测 → 图 → 1-hop 扩展。

- 无需重跑数据管道，`clone` 后直接运行（依赖：`pandas`、`pyarrow`、`tokenizers`、`numpy`）。
- 如果环境里有 `sentence-transformers` + `faiss`，会走真实的 **BGE** 检索；否则自动降级为 **hashing 词袋**（仅用于演示流程，指标只反映词面匹配，不代表 BGE 效果）。
- 每个小节末尾都用 `print` 输出关键数字，方便对照 `docs/` 里的报告。

**目录**

1. 数据契约与三层坐标系
2. 切分算法（句子打包 + overlap）
3. 检索（Dense RAG 骨架）
4. 评测（Hit@k / MRR）
5. 图结构（引文边 + 文内边）
6. 1-hop 图扩展（CG-RAG 的最小实现）
7. QASPER 篇内基线（读 M2 的结果）
8. 项目地图

## 0. 环境与数据清单

先确认工作目录和数据文件。所有数据都在 `data/processed/` 下，`data/raw/` 只保留 tokenizer 和 PubMedQA 原始 JSON。

In [1]:
import json
import os
import re
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
print("工作目录:", ROOT)

FILES = {
    "QASPER 段落": "data/processed/paragraphs.parquet",
    "QASPER 题库": "data/processed/qa.parquet",
    "QASPER chunks": "data/processed/chunks/size512_ov102_soft/chunks.parquet",
    "PubMedQA 引文图 chunks": "data/processed/pubmedqa_graph/chunks.parquet",
    "PubMedQA 引文图 题库": "data/processed/pubmedqa_graph/qa.parquet",
    "PubMedQA 引文图 边": "data/processed/pubmedqa_graph/graph_edges.parquet",
    "PubMedQA 引文图 论文": "data/processed/pubmedqa_graph/papers.parquet",
    "BGE tokenizer": "data/raw/bge_tok/tokenizer.json",
}
for name, p in FILES.items():
    print(f"{name:22s}", "OK " if os.path.exists(p) else "缺失", p)

print("\npandas", pd.__version__, "| numpy", np.__version__)
try:
    import sentence_transformers, faiss  # noqa: F401
    BACKEND = "bge"
except Exception:
    BACKEND = "hashing"
print("检索后端:", BACKEND, "(hashing 为无依赖降级，仅演示流程)")

工作目录: D:\Users\NTU\6405\project\RAG_Project
QASPER 段落              OK  data/processed/paragraphs.parquet
QASPER 题库              OK  data/processed/qa.parquet
QASPER chunks          OK  data/processed/chunks/size512_ov102_soft/chunks.parquet
PubMedQA 引文图 chunks    OK  data/processed/pubmedqa_graph/chunks.parquet
PubMedQA 引文图 题库        OK  data/processed/pubmedqa_graph/qa.parquet
PubMedQA 引文图 边         OK  data/processed/pubmedqa_graph/graph_edges.parquet
PubMedQA 引文图 论文        OK  data/processed/pubmedqa_graph/papers.parquet
BGE tokenizer          OK  data/raw/bge_tok/tokenizer.json

pandas 3.1.0 | numpy 2.5.4
检索后端: hashing (hashing 为无依赖降级，仅演示流程)


## 1. 数据契约与三层坐标系

项目里最容易被误用的一件事是**坐标系**：

```
论文扁平文本  --char_start/char_end-->  chunk
   └-- 段落 text_clean --para_spans[a:b]--> chunk 内的片段
```

- `para_id = "{paper_id}#p{idx:04d}"`，与切分配置无关，是全局稳定键。
- `chunk_id = "{paper_id}#s{sec}#c{idx:03d}"`，**只在同一配置内稳定**。
- `para_spans` 的偏移相对**段落自己**，不是相对 chunk；parquet 里存的是 JSON 文本。

In [2]:
pa = pd.read_parquet("data/processed/paragraphs.parquet")
qa = pd.read_parquet("data/processed/qa.parquet")
ck = pd.read_parquet("data/processed/chunks/size512_ov102_soft/chunks.parquet")

print("QASPER 段落:", len(pa), "| 题库:", len(qa), "| chunks:", len(ck))
print("题库分母(zero_gold_reason=='') :", int((qa.zero_gold_reason == "").sum()))

row = ck.iloc[0]
spans = json.loads(row.para_spans)
print("\n示例 chunk:", row.chunk_id)
print("  覆盖段落:", list(row.para_ids))
print("  段内偏移:", spans)
print("  不变量 len(para_spans)==len(para_ids):", len(spans) == len(row.para_ids))
print("  n_tokens:", int(row.n_tokens), "| section:", row.section_name)

QASPER 段落: 61317 | 题库: 3598 | chunks: 17653
题库分母(zero_gold_reason=='') : 2991

示例 chunk: 1503.00841#sabs#c000
  覆盖段落: ['1503.00841#p0000', '1503.00841#p0001', '1503.00841#p0002', '1503.00841#p0003', '1503.00841#p0004']
  段内偏移: [['1503.00841#p0000', 0, 606], ['1503.00841#p0001', 0, 406], ['1503.00841#p0002', 0, 677], ['1503.00841#p0003', 0, 625], ['1503.00841#p0004', 0, 255]]
  不变量 len(para_spans)==len(para_ids): True
  n_tokens: 496 | section: Abstract


## 2. 切分算法

核心就三件事：**按句切 → 按 token 预算打包 → 留 overlap**。
下面是 `src/rag_data/chunker.py` 的简化版（与脚本 `03_build_chunks.py` 同逻辑），
用一个真实的长段落演示切出来的边界。

In [3]:
from tokenizers import Tokenizer

TOK = Tokenizer.from_file("data/raw/bge_tok/tokenizer.json")
SENT_RE = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text):
    spans, start = [], 0
    for m in SENT_RE.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans or [(0, len(text))]

def pack_chunks(text, max_tokens=512, overlap=102):
    sents = split_sentences(text)
    out, i, n = [], 0, len(sents)
    while i < n:
        a = sents[i][0]
        if len(TOK.encode(text[a:sents[i][1]]).ids) > max_tokens:
            lo, hi = a, sents[i][1]
            while hi - lo > 1:
                mid = (lo + hi) // 2
                if len(TOK.encode(text[a:mid]).ids) <= max_tokens:
                    lo = mid
                else:
                    hi = mid
            b = lo
            out.append((a, b, len(TOK.encode(text[a:b]).ids), True))
            sents[i] = (b, sents[i][1])
            if b >= sents[i][1]:
                i += 1
            continue
        j = i
        while j + 1 < n and len(TOK.encode(text[a:sents[j + 1][1]]).ids) <= max_tokens:
            j += 1
        b = sents[j][1]
        out.append((a, b, len(TOK.encode(text[a:b]).ids), False))
        if j + 1 >= n:
            break
        k = j
        while k > i and len(TOK.encode(text[sents[k][0]:b]).ids) < overlap:
            k -= 1
        i = max(k, i + 1)
    return out

long_p = pa[pa.n_tokens > 512].iloc[0]
print(f"演示段落 {long_p.para_id} | {long_p.n_tokens} tok")
for a, b, t, tr in pack_chunks(long_p.text_clean):
    flag = " [硬切]" if tr else ""
    print(f"  [{a:5d},{b:5d}) {t:3d} tok{flag} | {long_p.text_clean[a:b][:55]}...")

演示段落 1910.14497#p0025 | 888 tok
  [    0, 1548) 512 tok [硬切] | 'accountant', 'acquaintance', 'actor', 'actress', 'admi...
  [ 1548, 2690) 380 tok | , 'marshal', 'mathematician', 'mechanic', 'midfielder',...


## 3. 检索（Dense RAG 骨架）

真实实现是 **BGE 编码 + FAISS `IndexFlatIP` 余弦检索**（见 `baselines/member2_dense_rag/`）。
这里为了让 notebook 在任何环境都能跑，封装成一个 `encode()`：

- 有 `sentence-transformers`：用 `BAAI/bge-small-en-v1.5`；
- 否则：用 **hashing 词袋**（CRC32 哈希到 4096 维，L2 归一化）。**只演示流程，不要拿它的数字当 BGE 效果**。

接着在 PubMedQA 引文图语料上建索引：**1,395 篇 / 3,707 chunks**。

In [4]:
g_chunks = pd.read_parquet("data/processed/pubmedqa_graph/chunks.parquet")
g_qa = pd.read_parquet("data/processed/pubmedqa_graph/qa.parquet")
g_papers = pd.read_parquet("data/processed/pubmedqa_graph/papers.parquet")
g_edges = pd.read_parquet("data/processed/pubmedqa_graph/graph_edges.parquet")

DIM = 4096
_enc = None

def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def encode(texts):
    global _enc
    if BACKEND == "bge" and _enc is None:
        from sentence_transformers import SentenceTransformer
        _enc = SentenceTransformer("BAAI/bge-small-en-v1.5")
    if BACKEND == "bge":
        return _enc.encode(texts, normalize_embeddings=True, batch_size=32)
    m = np.zeros((len(texts), DIM), dtype=np.float32)
    for r, t in enumerate(texts):
        for w in _tokens(t):
            m[r, zlib.crc32(w.encode()) % DIM] += 1.0
        n = np.linalg.norm(m[r])
        if n:
            m[r] /= n
    return m

t0 = time.time()
C = encode(g_chunks.text.tolist())
print("chunk 向量:", C.shape, f"| {time.time()-t0:.1f}s | backend={BACKEND}")

qid = "21645374"
qtext = g_qa[g_qa.question_id == qid].iloc[0].question
qv = encode([qtext])[0]
scores = C @ qv
top = np.argsort(-scores)[:5]
print("\n问题:", qtext)
for rank, i in enumerate(top, 1):
    r = g_chunks.iloc[i]
    print(f"  #{rank} {r.chunk_id} | {r.paper_id} | {r.section_name} | {scores[i]:.3f}")
    print(f"      {r.text[:70]}...")

chunk 向量: (3707, 4096) | 0.1s | backend=hashing

问题: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
  #1 W2082571460#s00#c000 | W2082571460 | PURPOSE | 0.324
      A possible role for fondaparinux as a bridging agent in the perioperat...
  #2 W2018063873#s00#c000 | W2018063873 | BACKGROUND | 0.322
      Programmed cell death (PCD) is the regulated death of cells within an ...
  #3 W2018063873#s01#c000 | W2018063873 | RESULTS | 0.274
      The following paper elucidates the role of mitochondrial dynamics duri...
  #4 W2094548557#s04#c000 | W2094548557 | MAIN OUTCOME MEASURE | 0.267
      Death....
  #5 W2262039363#s00#c000 | W2262039363 | BACKGROUND | 0.260
      Anchoring vignettes are brief texts describing a hypothetical characte...


## 4. 评测：Hit@k 与 MRR

PubMedQA 引文图的 gold 是**源论文的 chunk**（检索池是全文库，不按 paper 限定）：

$$\text{Hit@}k=\frac{1}{N}\sum_i \mathbb{1}[\text{top-}k_i \cap \text{gold}_i \neq \varnothing],\qquad
\text{MRR}=\frac{1}{N}\sum_i \frac{1}{\text{rank}_i}$$

下面在全量 **951 题**上跑一遍当前检索后端，给出真实数字。

In [5]:
gold = {r.question_id: set(r.gold_chunk_ids) for r in g_qa.itertuples()}
chunk_ids = g_chunks.chunk_id.tolist()
id2idx = {c: i for i, c in enumerate(chunk_ids)}
qids = list(gold)
Q = encode(g_qa.question.fillna("").tolist())

S = Q @ C.T

def hit_at(k, ranking):
    return np.mean([bool(set(ranking[i][:k]) & gold[q]) for i, q in enumerate(qids)])

def mrr_at(k, ranking):
    vals = []
    for i, q in enumerate(qids):
        r = ranking[i][:k]
        rr = 0.0
        for pos, c in enumerate(r, 1):
            if c in gold[q]:
                rr = 1.0 / pos
                break
        vals.append(rr)
    return float(np.mean(vals))

def rank_all(S):
    order = np.argsort(-S, axis=1)[:, :10]
    return [[chunk_ids[j] for j in row] for row in order]

t0 = time.time()
rk = rank_all(S)
print(f"检索 {len(qids)} 题 | {time.time()-t0:.1f}s | backend={BACKEND}")
print(f"Hit@1 {hit_at(1, rk):.4f} | Hit@3 {hit_at(3, rk):.4f} | Hit@10 {hit_at(10, rk):.4f} | MRR {mrr_at(10, rk):.4f}")
print("\n注: hashing 后端只反映词面重合；真实实验请用 M2 的 BGE 管道跑同一套指标。")

检索 951 题 | 0.1s | backend=hashing
Hit@1 0.6856 | Hit@3 0.7771 | Hit@10 0.8475 | MRR 0.7373

注: hashing 后端只反映词面重合；真实实验请用 M2 的 BGE 管道跑同一套指标。


## 5. 图结构：引文边 + 文内边

`graph_edges.parquet` 有两种边：

- `cites`（paper 级）：源论文 → 被引论文，**1,042 条**（只保留被引 ≥2 的“耦合核心”参考文献）；
- `adjacency`（chunk 级）：同篇相邻 chunk，**2,312 条**。

跨文档边是 CG-RAG 能工作的前提；文内边则撑起“篇内多 chunk”的结构。

In [6]:
print(g_edges.edge_type.value_counts().to_string())
print("\n节点规模: papers", g_papers.paper_id.nunique(), "| chunks", len(g_chunks))

cites = g_edges[g_edges.edge_type == "cites"]
deg = cites.groupby("src_id").size().sort_values(ascending=False)
pid = "W2147896631"
print(f"\n示例论文 {pid} 的出边: {int(deg.get(pid, 0))} 条")
for x in g_papers[g_papers.paper_id.isin(cites[cites.src_id == pid].dst_id)].head(5).itertuples():
    print("  ->", x.paper_id, "|", x.title[:60])

edge_type
adjacency    2312
cites        1042

节点规模: papers 1395 | chunks 3707

示例论文 W2147896631 的出边: 15 条
  -> W1927710737 | Obesity as a medical problem
  -> W1996699012 | Laparoscopic Sleeve Gastrectomy—Volume and Pressure Assessme
  -> W1993208462 | Gastric Leak After Laparoscopic-Sleeve Gastrectomy for Obesi
  -> W1985794584 | Reinforcing the Staple Line During Laparoscopic Sleeve Gastr
  -> W2022282990 | Long-term Results of Laparoscopic Sleeve Gastrectomy for Obe


## 6. 1-hop 图扩展（CG-RAG 的最小实现）

CG-RAG 的 LeSeGR 是训练出来的 Graph Transformer；但它的**机制骨架**可以先免训练地验证：

1. dense 取 top-10 作为种子；
2. 种子 chunk → 所属论文 → 沿 `cites` 边找邻居论文；
3. 邻居论文的 chunk 进入候选；
4. 重新打分：$\text{score}(q,c)=\text{sim}(q,c)+\lambda\cdot\max_{s\in\text{seeds}}\text{sim}(c,s)$。

如果这一步相比 dense 没有提升，就没必要上昂贵的图编码器——这是“先证伪再投入”的顺序。

In [7]:
paper_of = dict(zip(g_chunks.chunk_id, g_chunks.paper_id))
chunks_of = {}
for r in g_chunks.itertuples():
    chunks_of.setdefault(r.paper_id, []).append(r.chunk_id)

nbr = {}
for r in g_edges[g_edges.edge_type == "cites"].itertuples():
    nbr.setdefault(r.src_id, set()).add(r.dst_id)
    nbr.setdefault(r.dst_id, set()).add(r.src_id)

LAM = 0.5
BASE_K = 50

def expand_rank(i, topk=10, base_k=BASE_K):
    seed = [chunk_ids[j] for j in np.argsort(-S[i])[:topk]]
    pool = [chunk_ids[j] for j in np.argsort(-S[i])[:base_k]]
    papers = {paper_of[c] for c in seed}
    cand_papers = set(papers)
    for p in papers:
        cand_papers |= nbr.get(p, set())
    cand = {c for p in cand_papers for c in chunks_of.get(p, [])} | set(pool)
    ci = np.array([id2idx[c] for c in cand])
    base = S[i, ci]
    si = np.array([id2idx[c] for c in seed])
    neigh = (C[ci] @ C[si].T).max(axis=1)
    total = base + LAM * neigh
    order = ci[np.argsort(-total)]
    return [chunk_ids[j] for j in order[:topk]], len(cand)

t0 = time.time()
rk2, sizes = [], []
for i in range(len(qids)):
    r, n = expand_rank(i)
    rk2.append(r)
    sizes.append(n)
changed = sum(1 for a, b in zip(rk, rk2) if a != b)
recovered = sum(
    1
    for i, q in enumerate(qids)
    if not (set(rk[i][:10]) & gold[q]) and (set(rk2[i][:10]) & gold[q])
)
print(f"扩展完成 | {time.time()-t0:.1f}s | 平均候选数 {np.mean(sizes):.0f}（dense 候选池 50）")
print("top-10 结果发生变化的题:", changed, "/", len(qids), "| 因扩展而新召回 gold 的题:", recovered)

print("\n            Hit@1    Hit@3    Hit@10    MRR")
print(f"dense     {hit_at(1,rk):7.4f} {hit_at(3,rk):8.4f} {hit_at(10,rk):9.4f} {mrr_at(10,rk):7.4f}")
print(f"1-hop     {hit_at(1,rk2):7.4f} {hit_at(3,rk2):8.4f} {hit_at(10,rk2):9.4f} {mrr_at(10,rk2):7.4f}")
print("\n注: 排序有变化但新召回为 0，说明在这个后端上 1-hop 没带来增益——")
print("    这正是“先用廉价版证伪，再决定要不要上训练型 GNN”的意义；换 BGE 后需重跑。")

扩展完成 | 0.9s | 平均候选数 92（dense 候选池 50）
top-10 结果发生变化的题: 92 / 951 | 因扩展而新召回 gold 的题: 0

            Hit@1    Hit@3    Hit@10    MRR
dense      0.6856   0.7771    0.8475  0.7373
1-hop      0.6856   0.7771    0.8475  0.7374

注: 排序有变化但新召回为 0，说明在这个后端上 1-hop 没带来增益——
    这正是“先用廉价版证伪，再决定要不要上训练型 GNN”的意义；换 BGE 后需重跑。


## 7. QASPER 篇内基线（读 M2 的结果）

QASPER 是**篇内检索**：每题绑定一篇论文，候选池是该论文的 chunk。
M2 已经用 BGE + FAISS 跑完 dev（890 题有 gold），这里直接读它的结果，避免重复计算。

In [8]:
p = Path("baselines/member2_dense_rag/docs/qasper_dev_metrics.json")
if p.exists():
    d = json.loads(p.read_text(encoding="utf-8"))
    print("split:", d["split"], "| total:", d["total_questions"], "| eligible:", d["eligible_questions"])
    print("            Recall@k    MRR")
    for row in d["metrics"]:
        print(f"  k={row['k']:<2d}      {row['macro_paragraph_recall']*100:6.2f}%   {row['mrr']:.4f}")
else:
    print("缺少 M2 指标文件")

split: dev | total: 1005 | eligible: 890
            Recall@k    MRR
  k=1        33.70%   0.4213
  k=3        63.00%   0.5569
  k=5        77.33%   0.5836
  k=10       94.37%   0.6006


## 8. 项目地图

| notebook 小节 | 真实实现 |
|---|---|
| 数据契约 | `docs/contract/data_contract.md`、`src/rag_data/validate.py` |
| 切分算法 | `src/rag_data/chunker.py`、`scripts/03_build_chunks.py` |
| 切分实验 | `scripts/04_chunk_grid.py`、`docs/experiments/chunk_grid_results.md` |
| Dense 检索 | `baselines/member2_dense_rag/dense_rag/`（BGE + FAISS） |
| 评测 | `baselines/member2_dense_rag/dense_rag/retrieval_metrics.py` |
| 图构建 | `scripts/21_build_pubmedqa_graph.py`、`data/processed/pubmedqa_graph/graph_edges.parquet` |
| GNN / 图实验 | `data_process/`、`data_process/gnn_runs/` |
| 交互 demo | `demo/` |

**两个必须记住的口径**

1. QASPER 的 recall 分母是 **2,991**（dev 子集 890），不是 3,598；PubMedQA 引文图的分母是 **951**（49 个 PMID 未被 OpenAlex 收录）。
2. `chunk_id` 跨配置不稳定，跨切分比较一律落到 `para_id`。